# Agent Tools & Security

## Overview

This document explains how the debug assistant's four tools are designed — what each one does, why these four were chosen, and the security measures that prevent a runaway agent from reading files outside the project or executing arbitrary commands.

## Architecture Context

The debug assistant agent has four tools: `search_code` (semantic), `read_file` (exact), `grep` (pattern), and `run_tests` (verification). The LLM sees their JSON schema definitions at prompt time and decides which to call based on those descriptions. `tools.py` defines both the schemas and the execution functions. A single dispatcher, `execute_tool()`, routes by name.

**Why these four?** They map directly to how a human developer debugs:

1. *"Where might the bug be?"* → `search_code` — semantic similarity search over embedded code chunks in Qdrant
2. *"Let me actually see the code"* → `read_file` — read exact file content with line-range control
3. *"Where does this error message appear?"* → `grep` — regex match across the filesystem
4. *"Does my hypothesis hold?"* → `run_tests` — run pytest and get the result

Each tool has exactly one job. That specificity is intentional: narrow tools are easier to describe precisely in JSON schema, and a precise description produces better LLM decisions about when to use them.

**The security problem:** The agent is autonomous. Left unconstrained, it could be prompted (intentionally or via injection in a user's codebase) to read `/etc/passwd`, traverse outside the project root, or run arbitrary shell commands. The tools are the attack surface — securing them is the most important engineering decision in this service.

## Package Introductions

The tools for Qdrant, embeddings, and FastAPI are covered in earlier notebooks. This notebook introduces only the new packages.

### subprocess (stdlib)

Used to run pytest as a child process. The key API is `subprocess.run()`, which blocks until the process completes and returns stdout/stderr.

**Why subprocess instead of importing pytest directly?** Isolation. If you `import pytest` and call `pytest.main()`, the test run shares the same Python process as the debug service. A test that does something pathological — modifying global state, patching stdlib functions, triggering a segfault — can corrupt or crash the service. Running pytest as a subprocess keeps those side effects contained: the child process dies, the service lives.

### fnmatch (stdlib)

Used for glob-style pattern matching on filenames (e.g., `*.py`, `test_*.py`). Simple, zero dependencies. `fnmatch.fnmatch(filename, pattern)` returns a bool — that's the entire API we need.

### re (stdlib)

Used for regex matching inside the `grep` tool. The agent can search with regex patterns, not just literal strings — so a query like `r'def \w+_handler'` finds all handler functions in one pass. `re.compile()` pre-compiles the pattern; `pattern.search(line)` checks for a match anywhere in the line.

## Go/TS Comparison

| Concept | Go/TS | Python |
|---------|-------|--------|
| Run external process | `exec.Command("pytest", ".").Output()` | `subprocess.run(["pytest", "."], capture_output=True)` |
| Path traversal check | `filepath.Abs(p)` + `strings.HasPrefix(abs, root)` | `os.path.realpath(p)` + `str.startswith(root)` |
| Regex matching | `regexp.MustCompile(pattern).MatchString(line)` | `re.compile(pattern).search(line)` |
| Glob matching | `filepath.Match("*.py", name)` | `fnmatch.fnmatch(name, "*.py")` |

The path traversal defense is the most important row. In Go, `filepath.Abs` resolves `..` segments; `strings.HasPrefix` then checks the result. Python's `os.path.normpath` resolves `..` segments but **does not follow symlinks** — you must use `os.path.realpath` for full resolution. That difference is the Pitfall covered in Step 2 below.

## Build It

### Step 1: Design tool definitions

The LLM doesn't call Python functions directly. It reads JSON schema definitions and emits a structured JSON response saying which tool to call and with what arguments. Your code then executes the actual function.

The schema has three important parts:
- **`name`** — how the LLM references the tool in its response
- **`description`** — written for the LLM, not for humans. This is what the LLM reads to decide *when* to use the tool. Vague descriptions lead to poor tool selection.
- **`parameters`** — JSON Schema for the arguments. `required` lists the fields the LLM must always include; omitting a field from `required` makes it optional.

Here's the schema for `read_file` as an example:

In [ ]:
# Tool definition: read_file
# The LLM reads this schema and uses "description" to decide when to call the tool.
# Parameter descriptions tell the LLM what to fill in for each argument.

READ_FILE_TOOL = {
    "name": "read_file",
    "description": (
        "Read the contents of a specific file in the project. "
        "Use this when you know the file path and want to see the exact code. "
        "Optionally restrict to a line range to focus on a specific function or section."
    ),
    "parameters": {
        "type": "object",
        "properties": {
            "path": {
                "type": "string",
                "description": "Relative path from the project root (e.g., 'services/chat/app/chain.py')",
            },
            "start_line": {
                "type": "integer",
                "description": "First line to read (1-indexed, inclusive). Omit to start from the beginning.",
            },
            "end_line": {
                "type": "integer",
                "description": "Last line to read (inclusive). Omit to read to the end of the file.",
            },
        },
        "required": ["path"],
    },
}

# Only "path" is required. The LLM may optionally supply start_line and end_line
# when it wants to read a specific section rather than the whole file.
print("Required parameters:", READ_FILE_TOOL["parameters"]["required"])
print("Optional parameters:", [
    k for k in READ_FILE_TOOL["parameters"]["properties"]
    if k not in READ_FILE_TOOL["parameters"]["required"]
])

### Step 2: Implement read_file with security

The agent will be handed a `path` argument that came from an LLM. That path could be anything — including an attacker-crafted path designed to escape the project root.

The defense is a two-step check: resolve the path completely (following symlinks and collapsing `..`), then assert the result lives under the project root. If it doesn't, reject the call entirely.

In [ ]:
import os
import tempfile

# ── Set up a fake project root with some files ──────────────────────────────
tmp_root = tempfile.mkdtemp()
project_root = os.path.join(tmp_root, "my_project")
os.makedirs(os.path.join(project_root, "services", "chat"))

# Create a sample source file
sample_file = os.path.join(project_root, "services", "chat", "chain.py")
with open(sample_file, "w") as f:
    f.write("# chain.py\n")
    f.write("def build_chain(retriever, llm):\n")
    f.write("    return retriever | llm\n")

# Create a sensitive file OUTSIDE the project (simulates /etc/passwd)
secret_file = os.path.join(tmp_root, "sensitive.txt")
with open(secret_file, "w") as f:
    f.write("secret data\n")

print(f"Project root: {project_root}")
print(f"Secret file:  {secret_file}")

In [ ]:
# ── The security check ──────────────────────────────────────────────────────
# os.path.realpath:
#   - Resolves all '..' segments
#   - Follows symlinks to their final destination
#   - Returns an absolute path

def _safe_resolve(project_root: str, relative_path: str) -> str:
    """
    Resolve relative_path against project_root and verify it stays inside.
    Returns the absolute path if safe, raises ValueError if not.
    """
    # Build the candidate absolute path
    candidate = os.path.join(project_root, relative_path)

    # Fully resolve: collapse '..' and follow symlinks
    resolved = os.path.realpath(candidate)

    # The root itself must also be resolved (could be a symlink)
    safe_root = os.path.realpath(project_root)

    # Add trailing separator to prevent prefix collisions:
    # '/project_abc'.startswith('/project') would wrongly pass without it
    if not resolved.startswith(safe_root + os.sep) and resolved != safe_root:
        raise ValueError(f"Path traversal attempt blocked: {relative_path!r}")

    return resolved


# ── Test: legitimate path ───────────────────────────────────────────────────
try:
    safe = _safe_resolve(project_root, "services/chat/chain.py")
    print(f"ALLOWED: {safe}")
except ValueError as e:
    print(f"BLOCKED: {e}")

# ── Test: path traversal attack ─────────────────────────────────────────────
try:
    attack = _safe_resolve(project_root, "../../sensitive.txt")
    print(f"ALLOWED: {attack}")
except ValueError as e:
    print(f"BLOCKED: {e}")

# ── Test: absolute path smuggled in ─────────────────────────────────────────
try:
    attack2 = _safe_resolve(project_root, "/etc/passwd")
    print(f"ALLOWED: {attack2}")
except ValueError as e:
    print(f"BLOCKED: {e}")

In [ ]:
# ── Full read_file implementation ────────────────────────────────────────────

def read_file(
    project_root: str,
    path: str,
    start_line: int | None = None,
    end_line: int | None = None,
) -> str:
    """
    Read a file inside the project. Returns the content (or a slice of it).
    Returns an error string if the path is unsafe or the file is missing.
    """
    try:
        resolved = _safe_resolve(project_root, path)
    except ValueError as e:
        return f"Error: {e}"

    if not os.path.isfile(resolved):
        return f"Error: file not found: {path!r}"

    with open(resolved, "r", errors="replace") as f:
        lines = f.readlines()

    # Apply optional line range (1-indexed, inclusive)
    start = (start_line - 1) if start_line else 0
    end = end_line if end_line else len(lines)
    selected = lines[start:end]

    return "".join(selected)


# Read the whole file
result = read_file(project_root, "services/chat/chain.py")
print("=== Full file ===")
print(result)

# Read just line 2
result = read_file(project_root, "services/chat/chain.py", start_line=2, end_line=2)
print("=== Line 2 only ===")
print(result)

# Attack attempt returns an error string — agent sees this and can adjust
result = read_file(project_root, "../../sensitive.txt")
print("=== Attack attempt ===")
print(result)

> **Pitfall:** Don't use `os.path.normpath` alone — it resolves `..` segments but **does not follow symlinks**. A symlink at `project/link` that points to `/etc` would pass the prefix check with `normpath` because the path still looks like it's inside the project. `os.path.realpath` follows the symlink to its actual destination, so the check sees `/etc/...` and blocks it correctly. Always use `realpath` for security-sensitive path resolution.

### Step 3: Implement grep

The `grep` tool walks the project filesystem, compiles a regex, and returns matching lines with their file path and line number. Two design decisions keep it safe:

1. **Extension filter via fnmatch** — the caller can restrict search to `*.py` or `*.ts` files, avoiding noise from binary files or build artifacts.
2. **max_matches truncation** — without a cap, a broad pattern like `.` would return every line in the project, flooding the LLM context window. Truncation happens at collection time so the walk stops early.

In [ ]:
import re
import fnmatch
from pathlib import Path

# Add more files to the project so grep has something to walk
for name, content in [
    ("services/chat/retriever.py", "def retrieve(query):\n    pass\n"),
    ("services/ingestion/main.py", "def build_chain():\n    return None\n"),
]:
    path = os.path.join(project_root, name)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)


def grep(
    project_root: str,
    pattern: str,
    file_pattern: str = "*.py",
    max_matches: int = 50,
) -> str:
    """
    Search files matching file_pattern for lines matching the regex pattern.
    Returns up to max_matches results as formatted text.
    """
    try:
        compiled = re.compile(pattern)
    except re.error as e:
        return f"Error: invalid regex pattern: {e}"

    results = []
    safe_root = os.path.realpath(project_root)

    for dirpath, _dirnames, filenames in os.walk(safe_root):
        for filename in filenames:
            # fnmatch: simple glob-style filter on the filename only
            if not fnmatch.fnmatch(filename, file_pattern):
                continue

            filepath = os.path.join(dirpath, filename)
            # Make the path relative for readable output
            rel_path = os.path.relpath(filepath, safe_root)

            try:
                with open(filepath, "r", errors="replace") as f:
                    for lineno, line in enumerate(f, start=1):
                        if compiled.search(line):
                            results.append(f"{rel_path}:{lineno}: {line.rstrip()}")
                            if len(results) >= max_matches:
                                results.append(f"[truncated at {max_matches} matches]")
                                return "\n".join(results)
            except (OSError, PermissionError):
                continue  # skip unreadable files silently

    if not results:
        return f"No matches found for pattern: {pattern!r}"

    return "\n".join(results)


# Find all function definitions
print(grep(project_root, r"^def "))
print()

# Search for 'chain' in any Python file
print(grep(project_root, r"chain"))
print()

# Invalid regex returns an error string
print(grep(project_root, r"[invalid"))

### Step 4: Implement run_tests

Running tests is the most powerful tool — and the most constrained. The agent can only run pytest; it cannot run arbitrary shell commands. The implementation wraps subprocess with two hard limits: a timeout and output truncation.

In [ ]:
import subprocess


def run_tests(
    project_root: str,
    test_path: str = ".",
    timeout: int = 60,
    max_output_lines: int = 50,
) -> str:
    """
    Run pytest on test_path (relative to project_root).
    Returns stdout+stderr truncated to max_output_lines.
    """
    # Security: validate test_path stays inside the project before running
    try:
        resolved_test = _safe_resolve(project_root, test_path)
    except ValueError as e:
        return f"Error: {e}"

    cmd = ["python", "-m", "pytest", resolved_test, "-v", "--tb=short"]

    try:
        result = subprocess.run(
            cmd,
            cwd=project_root,
            capture_output=True,  # buffers stdout+stderr
            text=True,            # decode bytes → str automatically
            timeout=timeout,
        )
        output = result.stdout + result.stderr
    except subprocess.TimeoutExpired:
        # Return an error string — not an exception — so the agent can read it
        return f"Error: test run timed out after {timeout}s"
    except FileNotFoundError:
        return "Error: pytest not found. Is it installed in the current environment?"

    # Truncate output to keep it within the LLM context window
    lines = output.splitlines()
    if len(lines) > max_output_lines:
        lines = lines[:max_output_lines]
        lines.append(f"[output truncated at {max_output_lines} lines]")

    return "\n".join(lines)


# Demonstrate: run pytest on a directory with no tests
result = run_tests(project_root, ".", timeout=15)
print(result[:500])  # preview

> **Pitfall:** `subprocess.run` with `capture_output=True` buffers ALL output in memory before returning. A test suite that prints 10MB of logs will consume 10MB of RAM. The 50-line truncation happens *after* execution completes — it doesn't stop the process early. For this portfolio project that's acceptable. In a production tool you would use `subprocess.Popen` with streaming reads from `stdout` and a line counter that terminates the process once the limit is hit.

### Step 5: Build the dispatcher

`execute_tool()` is the bridge between the LLM's JSON response and the Python functions. When the LLM says `{"name": "read_file", "arguments": {"path": "main.py"}}`, the dispatcher unpacks and routes that call.

**Key design choice:** unknown tool names return an error *string*, not a raised exception. This matters because the agent loop catches the return value and passes it back to the LLM as an observation. If you raise an exception instead, the error never reaches the LLM — the agent loop dies, and the agent can't correct itself. By returning an error string, the agent sees what went wrong and can try a different tool.

In [ ]:
def execute_tool(project_root: str, tool_name: str, arguments: dict) -> str:
    """
    Dispatch a tool call from the LLM to the appropriate function.
    Always returns a string — either the tool result or an error message.
    """
    if tool_name == "read_file":
        return read_file(
            project_root,
            path=arguments["path"],
            start_line=arguments.get("start_line"),
            end_line=arguments.get("end_line"),
        )

    elif tool_name == "grep":
        return grep(
            project_root,
            pattern=arguments["pattern"],
            file_pattern=arguments.get("file_pattern", "*.py"),
            max_matches=arguments.get("max_matches", 50),
        )

    elif tool_name == "run_tests":
        return run_tests(
            project_root,
            test_path=arguments.get("test_path", "."),
            timeout=arguments.get("timeout", 60),
        )

    elif tool_name == "search_code":
        # search_code calls Qdrant (covered in notebooks 01-02).
        # Mocked here to keep this notebook self-contained.
        query = arguments.get("query", "")
        return f"[mock] semantic search results for: {query!r}"

    else:
        # Return error string — not raise. The agent reads this and adjusts.
        return f"Error: unknown tool {tool_name!r}. Available: read_file, grep, run_tests, search_code"


# ── Simulate the LLM calling tools ──────────────────────────────────────────

# LLM decides to read a file
call1 = {"name": "read_file", "arguments": {"path": "services/chat/chain.py"}}
print("=== read_file ===")
print(execute_tool(project_root, call1["name"], call1["arguments"]))

# LLM tries an unknown tool — gets an error string back
call2 = {"name": "execute_shell", "arguments": {"cmd": "rm -rf /"}}
print("=== unknown tool ===")
print(execute_tool(project_root, call2["name"], call2["arguments"]))

# LLM tries a path traversal via read_file
call3 = {"name": "read_file", "arguments": {"path": "../../sensitive.txt"}}
print("=== path traversal ===")
print(execute_tool(project_root, call3["name"], call3["arguments"]))

### Step 6: All four tool definitions together

Here is the complete `TOOL_DEFINITIONS` list as it appears in `tools.py`. Pay attention to the `description` fields — they are written *for the LLM*, not for human readers. The goal is to make the boundary between tools unambiguous so the model picks the right one.

In [ ]:
import json

TOOL_DEFINITIONS = [
    {
        "name": "search_code",
        "description": (
            "Semantic search over the codebase using vector similarity. "
            "Use this when you have a conceptual question or a natural-language description "
            "of what you're looking for (e.g., 'where is the retry logic for Ollama?'). "
            "Returns the most relevant code chunks from the vector database."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "Natural language description of the code you're looking for.",
                },
                "top_k": {
                    "type": "integer",
                    "description": "Number of results to return (default: 5).",
                },
            },
            "required": ["query"],
        },
    },
    {
        "name": "read_file",
        "description": (
            "Read the contents of a specific file in the project. "
            "Use this when you know the file path and want to see the exact code. "
            "Optionally restrict to a line range to focus on a specific function or section."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "path": {
                    "type": "string",
                    "description": "Relative path from the project root (e.g., 'services/chat/app/chain.py').",
                },
                "start_line": {
                    "type": "integer",
                    "description": "First line to read (1-indexed, inclusive). Omit to start from the beginning.",
                },
                "end_line": {
                    "type": "integer",
                    "description": "Last line to read (inclusive). Omit to read to the end of the file.",
                },
            },
            "required": ["path"],
        },
    },
    {
        "name": "grep",
        "description": (
            "Search for lines matching a regex pattern across all project files. "
            "Use this when you know a specific string, identifier, or error message "
            "that should appear literally in the code (e.g., 'ConnectionError', 'QDRANT_HOST'). "
            "Faster and more precise than search_code for exact matches."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "pattern": {
                    "type": "string",
                    "description": "Python regex pattern to search for (e.g., r'def \\w+_handler').",
                },
                "file_pattern": {
                    "type": "string",
                    "description": "Glob pattern to filter files (e.g., '*.py', 'test_*.py'). Defaults to '*.py'.",
                },
                "max_matches": {
                    "type": "integer",
                    "description": "Maximum number of matching lines to return (default: 50).",
                },
            },
            "required": ["pattern"],
        },
    },
    {
        "name": "run_tests",
        "description": (
            "Run the pytest test suite and return the results. "
            "Use this to verify whether a hypothesis is correct or to check if a bug has been fixed. "
            "Optionally target a specific test file or directory to run a subset of tests."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "test_path": {
                    "type": "string",
                    "description": "Relative path to a test file or directory (default: '.' runs all tests).",
                },
                "timeout": {
                    "type": "integer",
                    "description": "Maximum seconds to wait before aborting (default: 60).",
                },
            },
            "required": [],
        },
    },
]

# The LLM receives this list in the system prompt.
# Verify the structure is valid JSON.
print(f"Tools defined: {[t['name'] for t in TOOL_DEFINITIONS]}")
print(f"\nread_file description preview:")
print(TOOL_DEFINITIONS[1]['description'][:80] + "...")

## Experiment

### Experiment 1: Tool description quality affects LLM behavior

The `description` field is read by the LLM to decide *when* to call a tool. A vague description produces ambiguous tool selection — the model may reach for the wrong tool or ask for clarification unnecessarily. Compare these two descriptions for `grep`:

In [ ]:
# ── Vague description ────────────────────────────────────────────────────────
vague_grep = {
    "name": "grep",
    "description": "Search files.",
    # Problem: "search files" is identical in scope to search_code.
    # The LLM has no signal about whether to use grep or search_code
    # for a query like 'find where ConnectionError is handled'.
    # It might pick either — or alternate between them across sessions.
}

# ── Clear description ────────────────────────────────────────────────────────
clear_grep = {
    "name": "grep",
    "description": (
        "Search for lines matching a regex pattern across all project files. "
        "Use this when you know a specific string, identifier, or error message "
        "that should appear literally in the code (e.g., 'ConnectionError', 'QDRANT_HOST'). "
        "Faster and more precise than search_code for exact matches."
    ),
    # The phrase 'should appear literally in the code' draws the boundary:
    # grep = exact text match, search_code = conceptual similarity.
    # The phrase 'Faster and more precise than search_code' tells the model
    # to prefer grep when it has an exact string to look for.
}

print("Vague (18 chars):", repr(vague_grep["description"]))
print()
print("Clear (283 chars):", repr(clear_grep["description"]))
print()
print("Key phrases in the clear description:")
key_phrases = [
    "specific string, identifier, or error message",
    "appear literally in the code",
    "Faster and more precise than search_code",
]
for phrase in key_phrases:
    print(f"  ✓ {phrase!r}")

The signal `"Faster and more precise than search_code for exact matches"` is written specifically to break the tie: when the agent has a literal string to find, it should pick `grep`. When it has a concept, it should pick `search_code`. Without that disambiguation, the LLM falls back on its training priors — and might consistently reach for whichever tool name sounds more familiar.

### Experiment 2: What the agent sees when run_tests times out

A long-running or hung test suite should not block the agent indefinitely. The timeout turns `TimeoutExpired` into an error string the agent can act on.

In [ ]:
import time

# Create a test that sleeps longer than our timeout
slow_test_dir = os.path.join(project_root, "slow_tests")
os.makedirs(slow_test_dir, exist_ok=True)

slow_test_file = os.path.join(slow_test_dir, "test_slow.py")
with open(slow_test_file, "w") as f:
    f.write("import time\n")
    f.write("\n")
    f.write("def test_that_sleeps():\n")
    f.write("    time.sleep(5)  # sleeps 5 seconds\n")
    f.write("    assert True\n")

print("Running slow test with 2-second timeout...")
start = time.time()
result = run_tests(project_root, "slow_tests", timeout=2)
elapsed = time.time() - start

print(f"Returned after {elapsed:.1f}s (not 5s)")
print(f"Agent sees: {result!r}")
print()
print("The agent receives this error string and can:")
print("  - Report the timeout to the user")
print("  - Suggest which test might be causing it (using grep or read_file)")
print("  - Try running a smaller subset of tests")

In [ ]:
# ── Cleanup tempfiles ────────────────────────────────────────────────────────
import shutil
shutil.rmtree(tmp_root)
print("Cleaned up temporary project directory.")

## Check Your Understanding

1. **Why do tool functions return error strings instead of raising exceptions? How does this affect the agent's behavior?**

2. **The `grep` tool walks the filesystem on every call rather than using a pre-built index. When would this become a problem, and what would you change?**

3. **Why is `search_code` (semantic) needed when `grep` (exact match) exists? Give an example query where one succeeds and the other fails.**